<a href="https://colab.research.google.com/github/sharancode3/AI-LAB-2026/blob/main/1BM24CS264_WEEK6_First_Order_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Week 6 - First Order Logic Resolution
# Aim: Prove queries using Resolution

objects = ["Apple", "Vegetables", "Peanuts", "Mango"]

# Knowledge Base in grounded CNF
KB = []

# C1: ~food(x) | likes(John,x)
for x in objects:
    KB.append(frozenset({
        f"~food({x})",
        f"likes(John,{x})"
    }))

# Facts
KB.append(frozenset({"food(Apple)"}))
KB.append(frozenset({"food(Vegetables)"}))
KB.append(frozenset({"eats(Anil,Peanuts)"}))
KB.append(frozenset({"alive(Anil)"}))

# C4: ~eats(y,z) | killed(y) | food(z)
for z in objects:
    KB.append(frozenset({
        f"~eats(Anil,{z})",
        "killed(Anil)",
        f"food({z})"
    }))

# C7: ~eats(Anil,w) | eats(Harry,w)
for w in objects:
    KB.append(frozenset({
        f"~eats(Anil,{w})",
        f"eats(Harry,{w})"
    }))

# C8 and C9
KB.append(frozenset({"killed(Anil)", "alive(Anil)"}))
KB.append(frozenset({"~alive(Anil)", "~killed(Anil)"}))


def complement(literal):
    if literal.startswith("~"):
        return literal[1:]
    return "~" + literal


def format_clause(clause):
    if not clause:
        return "{}"
    return " | ".join(sorted(clause))


def resolve(c1, c2):
    results = []

    for literal in sorted(c1):
        opposite = complement(literal)

        if opposite in c2:
            resolvent = frozenset(
                (set(c1) - {literal}) |
                (set(c2) - {opposite})
            )

            # Ignore tautological clauses
            if not any(
                complement(x) in resolvent
                for x in resolvent
            ):
                results.append((resolvent, literal, opposite))

    return results


def prove(query):
    negated_query = frozenset({"~" + query})

    known = list(KB)
    known.append(negated_query)

    support = [negated_query]
    parents = {}
    history = []

    while support:
        current = support.pop(0)

        for other in known.copy():
            if current == other:
                continue

            for result, literal, opposite in resolve(current, other):

                if not result:
                    parents[result] = (
                        current, other, literal, opposite
                    )
                    return True, parents, history

                if result not in known:
                    known.append(result)
                    support.append(result)

                    parents[result] = (
                        current, other, literal, opposite
                    )

                    history.append((
                        current, other, literal, opposite, result
                    ))

    return False, parents, history


def collect_proof(clause, parents, visited, proof):
    if clause in visited or clause not in parents:
        return

    visited.add(clause)

    c1, c2, literal, opposite = parents[clause]

    collect_proof(c1, parents, visited, proof)
    collect_proof(c2, parents, visited, proof)

    proof.append((c1, c2, literal, opposite, clause))


def display_step(i, c1, c2, literal, opposite, result):
    print(f"\nStep {i}:")
    print(" Clause 1:", format_clause(c1))
    print(" Clause 2:", format_clause(c2))
    print(" Complementary literals:", literal, "and", opposite)
    print(" Resolvent:", format_clause(result))


def run_test(number, query):
    print("\n" + "=" * 45)
    print("TEST CASE", number)
    print("Query:", query)
    print("Negated Query: ~" + query)

    success, parents, history = prove(query)

    if success:
        proof = []

        collect_proof(
            frozenset(), parents, set(), proof
        )

        print("\nRESOLUTION STEPS")

        for i, step in enumerate(proof, 1):
            display_step(i, *step)

        print("\nEmpty clause {} obtained.")
        print("RESULT: Query Proved.")

    else:
        print("\nRESOLUTION STEPS ATTEMPTED")

        for i, step in enumerate(history, 1):
            display_step(i, *step)

        print("\nNo empty clause obtained.")
        print("RESULT: Query Not Proved.")


# Test Case 1: Successful proof
run_test(1, "likes(John,Peanuts)")

# Test Case 2: Unsuccessful proof
run_test(2, "likes(John,Mango)")


TEST CASE 1
Query: likes(John,Peanuts)
Negated Query: ~likes(John,Peanuts)

RESOLUTION STEPS

Step 1:
 Clause 1: ~likes(John,Peanuts)
 Clause 2: likes(John,Peanuts) | ~food(Peanuts)
 Complementary literals: ~likes(John,Peanuts) and likes(John,Peanuts)
 Resolvent: ~food(Peanuts)

Step 2:
 Clause 1: ~food(Peanuts)
 Clause 2: food(Peanuts) | killed(Anil) | ~eats(Anil,Peanuts)
 Complementary literals: ~food(Peanuts) and food(Peanuts)
 Resolvent: killed(Anil) | ~eats(Anil,Peanuts)

Step 3:
 Clause 1: killed(Anil) | ~eats(Anil,Peanuts)
 Clause 2: eats(Anil,Peanuts)
 Complementary literals: ~eats(Anil,Peanuts) and eats(Anil,Peanuts)
 Resolvent: killed(Anil)

Step 4:
 Clause 1: killed(Anil)
 Clause 2: ~alive(Anil) | ~killed(Anil)
 Complementary literals: killed(Anil) and ~killed(Anil)
 Resolvent: ~alive(Anil)

Step 5:
 Clause 1: ~alive(Anil)
 Clause 2: alive(Anil)
 Complementary literals: ~alive(Anil) and alive(Anil)
 Resolvent: {}

Empty clause {} obtained.
RESULT: Query Proved.

TEST CASE 2